In [1]:
import pandas as pd
import numpy as np
import os
import re
import asyncio
import aiohttp
import nest_asyncio
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
import catboost
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import roc_auc_score
import time

nest_asyncio.apply()

DIR = '/kaggle/input/pump-fun-graduation-february-2025'
TRAIN_PATH = os.path.join(DIR, 'train.csv')
TEST_PATH  = os.path.join(DIR, 'test_unlabeled.csv')
DUNE_INFO_PATH = os.path.join(DIR, 'dune_token_info_v2.csv')
TOKEN_INFO_PATH = os.path.join(DIR, 'token_info_onchain_divers_v2.csv')

print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
dune  = pd.read_csv(DUNE_INFO_PATH)
token = pd.read_csv(TOKEN_INFO_PATH)
print("Data loaded.")

print("Processing metadata...")
token_info_features = token[['mint', 'creator', 'bundle_size', 'gas_used', 'name', 'symbol', 'url']].copy()
token_info_features['bundle_size'] = token_info_features['bundle_size'].fillna(1).astype(int)
token_info_features['gas_used'] = token_info_features['gas_used'].fillna(token_info_features['gas_used'].median())
creator_counts = token_info_features['creator'].value_counts().to_dict()
token_info_features['creator_token_count'] = token_info_features['creator'].map(creator_counts)
token_info_features['creator_gas_avg'] = token_info_features.groupby('creator')['gas_used'].transform('mean')
creator_map = token_info_features.set_index('mint')['creator'].to_dict()

dune_info_features = dune[['token_mint_address', 'decimals', 'created_at', 'token_uri']].copy()
dune_info_features.rename(columns={'token_mint_address': 'mint'}, inplace=True)
dune_info_features['decimals'] = dune_info_features['decimals'].fillna(0).astype(int)
dune_info_features['created_at'] = pd.to_datetime(dune_info_features['created_at'], utc=True, errors='coerce')
dune_info_features['launch_hour'] = dune_info_features['created_at'].dt.hour
dune_info_features['launch_day'] = dune_info_features['created_at'].dt.dayofweek
print("Metadata processed.")

def process_chunks(files, creator_mapping):
    if not files:
        raise RuntimeError("No chunk files found—check your directory.")
    feats = []
    for f in tqdm(files, desc='Processing chunks'):
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f"Error reading {f}: {e}. Skipping.")
            continue

        required_cols = [
            'base_coin', 'block_time', 'quote_coin_amount', 'base_coin_amount',
            'direction', 'signing_wallet', 'virtual_sol_balance_after',
            'virtual_token_balance_after', 'consumed_gas', 'provided_gas_limit',
            'provided_gas_fee'
        ]
        if not all(col in df.columns for col in required_cols):
            print(f"Skipping {f} due to missing required columns.")
            continue

        df['creator'] = df['base_coin'].map(creator_mapping)
        df['block_time'] = pd.to_datetime(df['block_time'], utc=True, errors='coerce')
        df.dropna(subset=['block_time'], inplace=True)
        if df.empty:
            continue
        df = df.sort_values(by='block_time')

        df['price'] = df['quote_coin_amount'] / (df['base_coin_amount'] + 1e-9)
        df['time_diff'] = df.groupby('base_coin')['block_time'].diff().dt.total_seconds().fillna(0)
        df['gas_ratio'] = df['consumed_gas'] / (df['provided_gas_limit'] + 1)

        creator_actions = df[df['signing_wallet'] == df['creator']]
        creator_sells = creator_actions[creator_actions['direction'] == 'sell']
        creator_agg = {}
        if not creator_actions.empty:
            creator_agg['creator_tx_count'] = len(creator_actions)
            creator_agg['creator_sell_count'] = len(creator_sells)
            creator_agg['creator_buy_count'] = creator_agg['creator_tx_count'] - creator_agg['creator_sell_count']
            creator_agg['creator_sell_ratio'] = creator_agg['creator_sell_count'] / (creator_agg['creator_tx_count'] + 1e-9)
            if not creator_sells.empty:
                creator_agg['creator_did_sell'] = 1
                creator_agg['creator_total_sold_sol'] = creator_sells['quote_coin_amount'].sum()
                creator_agg['creator_avg_sell_sol'] = creator_sells['quote_coin_amount'].mean()
                creator_agg['creator_first_sell_time'] = creator_sells['block_time'].min()
                creator_agg['creator_last_sell_time'] = creator_sells['block_time'].max()
            else:
                creator_agg.update({
                    'creator_did_sell': 0,
                    'creator_total_sold_sol': 0,
                    'creator_avg_sell_sol': 0,
                    'creator_first_sell_time': pd.NaT,
                    'creator_last_sell_time': pd.NaT
                })
        else:
            for k in [
                'creator_tx_count','creator_sell_count','creator_buy_count','creator_sell_ratio',
                'creator_did_sell','creator_total_sold_sol','creator_avg_sell_sol'
            ]:
                creator_agg[k] = 0
            creator_agg['creator_first_sell_time'] = pd.NaT
            creator_agg['creator_last_sell_time'] = pd.NaT

        agg = df.groupby('base_coin').agg(
            total_sol_volume=('quote_coin_amount', 'sum'),
            avg_sol_volume=('quote_coin_amount', 'mean'),
            std_sol_volume=('quote_coin_amount', 'std'),
            max_sol_volume=('quote_coin_amount', 'max'),
            min_sol_volume=('quote_coin_amount', 'min'),
            p90_sol_volume=('quote_coin_amount', lambda x: np.quantile(x, 0.9) if len(x) else 0),
            count_buys=('direction', lambda x: (x=='buy').sum()),
            count_sells=('direction', lambda x: (x=='sell').sum()),
            buy_ratio=('direction', lambda x: (x=='buy').mean() if len(x) else 0.5),
            unique_wallets=('signing_wallet', 'nunique'),
            wallet_diversity=('signing_wallet', lambda x: x.nunique()/(len(x)+1e-9) if len(x) else 0),
            first_transaction_time=('block_time', 'min'),
            last_transaction_time=('block_time', 'max'),
            initial_liquidity_sol=('virtual_sol_balance_after', 'first'),
            final_liquidity_sol=('virtual_sol_balance_after', 'last'),
            std_liquidity_sol=('virtual_sol_balance_after', 'std'),
            initial_liquidity_token=('virtual_token_balance_after', 'first'),
            final_liquidity_token=('virtual_token_balance_after', 'last'),
            initial_price=('price', 'first'),
            final_price=('price', 'last'),
            avg_price=('price', 'mean'),
            price_volatility=('price', 'std'),
            max_price=('price', 'max'),
            min_price=('price', 'min'),
            avg_time_diff=('time_diff', 'mean'),
            std_time_diff=('time_diff', 'std'),
            min_time_diff=('time_diff', 'min'),
            max_time_diff=('time_diff', 'max'),
            avg_gas_ratio=('gas_ratio', 'mean'),
            std_gas_ratio=('gas_ratio', 'std'),
            avg_gas_fee=('provided_gas_fee', 'mean'),
            total_gas_fee=('provided_gas_fee', 'sum'),
            avg_consumed_gas=('consumed_gas', 'mean'),
            total_consumed_gas=('consumed_gas', 'sum')
        )

        agg = agg.merge(
            pd.DataFrame([creator_agg], index=[df['base_coin'].iloc[0]]),
            left_index=True, right_index=True, how='left'
        )

        agg['price_change'] = agg['final_price'] / (agg['initial_price'] + 1e-9)
        agg['liquidity_change_sol'] = agg['final_liquidity_sol'] - agg['initial_liquidity_sol']
        agg['liquidity_change_token'] = agg['final_liquidity_token'] - agg['initial_liquidity_token']
        agg['time_duration'] = (agg['last_transaction_time'] - agg['first_transaction_time']).dt.total_seconds()
        agg['tx_per_second'] = (agg['count_buys'] + agg['count_sells']) / (agg['time_duration'] + 1)
        agg['creator_first_sell_time_diff'] = (agg['creator_first_sell_time'] - agg['first_transaction_time']).dt.total_seconds()
        agg['creator_last_sell_time_diff'] = (agg['creator_last_sell_time'] - agg['first_transaction_time']).dt.total_seconds()
        agg['creator_first_sell_relative'] = agg['creator_first_sell_time_diff'] / (agg['time_duration'] + 1e-9)
        agg['creator_last_sell_relative'] = agg['creator_last_sell_time_diff'] / (agg['time_duration'] + 1e-9)

        for col in ['std_sol_volume','std_liquidity_sol','price_volatility','std_time_diff','std_gas_ratio']:
            agg[col] = agg[col].fillna(0)

        agg = agg.reset_index().rename(columns={'base_coin':'mint'})
        feats.append(agg)

    if not feats:
        raise RuntimeError("No features could be generated from chunks.")
    all_feats = pd.concat(feats, ignore_index=True)
    return all_feats.groupby('mint').agg('first').reset_index()

print("Processing transaction chunks...")
chunk_files = [os.path.join(DIR, f) for f in os.listdir(DIR) if f.startswith('chunk') and f.endswith('.csv')]
transaction_features = process_chunks(chunk_files, creator_map)
print("Transaction features processed.")

def merge_all(df):
    df = df.merge(transaction_features, on='mint', how='left')
    df = df.merge(token_info_features, on='mint', how='left')
    df = df.merge(dune_info_features, on='mint', how='left')
    return df

print("Merging features...")
train_m = merge_all(train)
test_m  = merge_all(test)
print("Features merged.")

print("Starting IPFS metadata retrieval...")
async def fetch_metadata(session, url, semaphore, timeout=1, retries=0):
    async with semaphore:
        for attempt in range(retries+1):
            try:
                async with session.get(url, timeout=timeout) as response:
                    if response.status == 200:
                        content_type = response.headers.get('Content-Type','')
                        if 'application/json' in content_type:
                            return await response.json()
                        else:
                            return {'error':'non_json_response','status':response.status}
                    else:
                        return {'error':'http_error','status':response.status}
            except asyncio.TimeoutError:
                if attempt == retries:
                    return {'error':'timeout'}
                await asyncio.sleep(0.5)
            except aiohttp.ClientError as e:
                return {'error':'client_error','exception':str(e)}
            except Exception as e:
                return {'error':'unknown','exception':str(e)}
        return {'error':'max_retries_exceeded'}

async def process_urls(url_list, concurrent_limit=100):
    semaphore = asyncio.Semaphore(concurrent_limit)
    results = {}
    url_list_unique = [u for u in pd.Series(url_list).dropna().unique() if isinstance(u,str) and u.startswith('http')]
    total = len(url_list_unique)
    if total == 0:
        print("No valid URLs found to process.")
        return {}
    processed = 0
    interval = max(1, total//20)
    async with aiohttp.ClientSession() as session:
        tasks = {u: asyncio.ensure_future(fetch_metadata(session,u,semaphore)) for u in url_list_unique}
        start = time.time()
        for u, fut in tasks.items():
            try:
                res = await fut
                results[u] = res
                processed += 1
                if processed % interval == 0 or processed==total:
                    elapsed = time.time()-start
                    print(f"Processed {processed}/{total} URLs ({processed/total:.1%}) in {elapsed:.2f}s")
            except Exception as e:
                print(f"Error on {u}: {e}")
                results[u] = {'error':'gather_exception','exception':str(e)}
    return results

async def process_all_ipfs(urls_token, urls_dune):
    print("Token URLs...")
    rt = await process_urls(urls_token)
    print("TokenURI URLs...")
    rd = await process_urls(urls_dune)
    print("IPFS done.")
    return rt, rd

token_urls = pd.Series(train_m['url'].tolist()+test_m['url'].tolist()).dropna().unique().tolist()
dune_urls  = pd.Series(train_m['token_uri'].tolist()+test_m['token_uri'].tolist()).dropna().unique().tolist()
try:
    loop = asyncio.get_event_loop()
    if loop.is_running():
        results_token, results_dune = asyncio.run(process_all_ipfs(token_urls,dune_urls))
    else:
        results_token, results_dune = loop.run_until_complete(process_all_ipfs(token_urls,dune_urls))
except RuntimeError:
    results_token, results_dune = asyncio.run(process_all_ipfs(token_urls,dune_urls))

def extract_ipfs_features(metadata):
    default = {'twitter':0,'website':0,'telegram':0,'discord':0,'desc_len':0,'num_keys':0,'description':''}
    if not isinstance(metadata,dict) or not metadata or 'error' in metadata:
        return default
    t = 1 if metadata.get('twitter') or metadata.get('x') else 0
    w = 1 if metadata.get('website') or metadata.get('extensions',{}).get('website') else 0
    tg= 1 if metadata.get('telegram') or metadata.get('tg') else 0
    d = 1 if metadata.get('discord') or metadata.get('dc') else 0
    desc = metadata.get('description','') or ''
    dl = len(desc)
    nk = len(metadata.keys())
    return {'twitter':t,'website':w,'telegram':tg,'discord':d,'desc_len':dl,'num_keys':nk,'description':desc}

ipfs_token = {u:extract_ipfs_features(r) for u,r in results_token.items()}
ipfs_dune  = {u:extract_ipfs_features(r) for u,r in results_dune.items()}

def add_ipfs(df,col,prefix,dmap):
    default=extract_ipfs_features({})
    feats=df[col].apply(lambda x:dmap.get(x,default) if pd.notna(x) else default)
    df[f'{prefix}_twitter']=feats.apply(lambda x:x['twitter'])
    df[f'{prefix}_website']=feats.apply(lambda x:x['website'])
    df[f'{prefix}_telegram']=feats.apply(lambda x:x['telegram'])
    df[f'{prefix}_discord']=feats.apply(lambda x:x['discord'])
    df[f'{prefix}_desc_len']=feats.apply(lambda x:x['desc_len'])
    df[f'{prefix}_num_keys']=feats.apply(lambda x:x['num_keys'])
    df[f'{prefix}_description']=feats.apply(lambda x:x['description'])
    return df

train_m = add_ipfs(train_m,'url','ipfs_url',ipfs_token)
test_m  = add_ipfs(test_m,'url','ipfs_url',ipfs_token)
train_m = add_ipfs(train_m,'token_uri','ipfs_tokenuri',ipfs_dune)
test_m  = add_ipfs(test_m,'token_uri','ipfs_tokenuri',ipfs_dune)

for df in [train_m,test_m]:
    df['combined_description'] = df['ipfs_tokenuri_description'].fillna('')
    mask = df['combined_description']==''
    df.loc[mask,'combined_description'] = df.loc[mask,'ipfs_url_description'].fillna('')
    df['combined_description']=df['combined_description'].str.lower().astype(str)
    df['combined_desc_len']=df['combined_description'].str.len()

keywords_desc=['utility','roadmap','airdrop','community','stake','reward','burn','tax','presale','launch']
pos_words=['good','great','strong','hold','hodl','diamond','pump','moon']
neg_words=['scam','rug','dump','sell','risk','warning']
for df in [train_m,test_m]:
    for kw in keywords_desc:
        df[f'desc_has_{kw}']=df['combined_description'].str.contains(kw,regex=False).astype(int)
    df['desc_word_count']=df['combined_description'].str.split().str.len().fillna(0)
    df['desc_has_link']=df['combined_description'].str.contains('http:|https:',regex=True).astype(int)
    df['desc_pos_score']=df['combined_description'].apply(lambda x: sum(x.count(w) for w in pos_words))
    df['desc_neg_score']=df['combined_description'].apply(lambda x: sum(x.count(w) for w in neg_words))

for df in [train_m,test_m]:
    df['name']=df['name'].fillna('').astype(str)
    df['symbol']=df['symbol'].fillna('').astype(str)
    df['name_len']=df['name'].str.len()
    df['symbol_len']=df['symbol'].str.len()
    df['name_word_count']=df['name'].str.split().str.len()
    df['name_upper_ratio']=df['name'].apply(lambda x: sum(1 for c in x if c.isupper())/(len(x)+1e-9))
    df['symbol_is_allcaps']=df['symbol'].apply(lambda x: x.isupper() and len(x)>1).astype(int)
    df['name_special_chars']=df['name'].apply(lambda x: len(re.findall(r'[^A-Za-z0-9 ]',x)))
    df['symbol_special_chars']=df['symbol'].apply(lambda x: len(re.findall(r'[^A-Za-z0-9]',x)))
    memes=['moon','ape','pump','doge','coin','mem','shib','flok','elon','sol','cat','pepe']
    for kw in memes:
        df[f'name_has_{kw}']=df['name'].str.lower().str.contains(kw).astype(int)
        df[f'symbol_has_{kw}']=df['symbol'].str.lower().str.contains(kw).astype(int)
    emoji_pattern=re.compile(
        r'[\U0001F600-\U0001F64F'
        r'\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF'
        r'\U0001F700-\U0001F77F'
        r'\U0001F780-\U0001F7FF'
        r'\U0001F800-\U0001F8FF'
        r'\U0001F900-\U0001F9FF'
        r'\U0001FA00-\U0001FA6F'
        r'\U0001FA70-\U0001FAFF'
        r'\U00002702-\U000027B0]+',flags=re.UNICODE)
    df['name_emoji_count']=df['name'].apply(lambda x: len(emoji_pattern.findall(x)))
    df['symbol_emoji_count']=df['symbol'].apply(lambda x: len(emoji_pattern.findall(x)))
    df['name_has_emoji']=df['name_emoji_count'].gt(0).astype(int)
    df['symbol_has_emoji']=df['symbol_emoji_count'].gt(0).astype(int)
    df['combined_text']=df['name']+' '+df['symbol']+' '+df['combined_description']

tfidf=TfidfVectorizer(
    max_features=500,stop_words='english',ngram_range=(1,2),
    analyzer='char_wb',max_df=0.9,min_df=3
)
full_text=pd.concat([train_m['combined_text'],test_m['combined_text']]).fillna('')
if full_text.str.strip().eq('').all():
    tfidf_svd_cols=[]
    for i in range(20):
        train_m[f'tfidf_svd_{i}']=0
        test_m[f'tfidf_svd_{i}']=0
else:
    try:
        X_txt=tfidf.fit_transform(full_text)
        n_comp=min(20,X_txt.shape[1]-1)
        if n_comp>0:
            svd=TruncatedSVD(n_components=n_comp,random_state=42)
            X_svd=svd.fit_transform(X_txt)
            tfidf_svd_cols=[]
            tr_len=len(train_m)
            for i in range(n_comp):
                train_m[f'tfidf_svd_{i}']=X_svd[:tr_len,i]
                test_m [f'tfidf_svd_{i}']=X_svd[tr_len:,i]
                tfidf_svd_cols.append(f'tfidf_svd_{i}')
        else:
            tfidf_svd_cols=[]
            for i in range(20):
                train_m[f'tfidf_svd_{i}']=0
                test_m[f'tfidf_svd_{i}']=0
    except ValueError:
        tfidf_svd_cols=[]
        for i in range(20):
            train_m[f'tfidf_svd_{i}']=0
            test_m[f'tfidf_svd_{i}']=0

meme_file=os.path.join(DIR,'meme_keywords.txt')
extended_memes=memes
if os.path.exists(meme_file):
    try:
        with open(meme_file) as f:
            ext=[l.strip().lower() for l in f if l.strip()]
        extended_memes=list(set(ext+memes))
    except:
        extended_memes=memes
for df in [train_m,test_m]:
    df['meme_score_name']=df['name'].str.lower().apply(lambda x: sum(x.count(kw) for kw in extended_memes))
    df['meme_score_symbol']=df['symbol'].str.lower().apply(lambda x: sum(x.count(kw) for kw in extended_memes))
    df['meme_score_desc']=df['combined_description'].apply(lambda x: sum(x.count(kw) for kw in extended_memes))
    df['has_meme_keyword_name']=df['meme_score_name'].gt(0).astype(int)
    df['has_meme_keyword_symbol']=df['meme_score_symbol'].gt(0).astype(int)
    df['has_meme_keyword_desc']=df['meme_score_desc'].gt(0).astype(int)
    df['has_meme_keyword_any']=(df['has_meme_keyword_name']|df['has_meme_keyword_symbol']|df['has_meme_keyword_desc']).astype(int)

for df in [train_m,test_m]:
    df['first_transaction_time']=pd.to_datetime(df['first_transaction_time'],utc=True,errors='coerce')
    df['created_at']=pd.to_datetime(df['created_at'],utc=True,errors='coerce')
    df['launch_time_gap']=(df['first_transaction_time']-df['created_at']).dt.total_seconds()
    df['liquidity_volatility_ratio']=df['std_liquidity_sol']/(df['avg_sol_volume']+1)
    df['price_stability']=1/(df['price_volatility'].fillna(0)+0.01)
    df['gas_efficiency']=df['total_consumed_gas']/(df['total_sol_volume']+1)
    df['buy_sell_ratio']=df['count_buys']/(df['count_sells']+1)
    df['unique_wallet_per_tx']=df['unique_wallets']/(df['count_buys']+df['count_sells']+1)
    df['sol_volume_per_wallet']=df['total_sol_volume']/(df['unique_wallets']+1)
    df['price_range']=df['max_price']-df['min_price']
    df['price_range_norm']=df['price_range']/(df['avg_price']+1e-9)
    df['creator_sell_impact']=df['creator_total_sold_sol']/(df['total_sol_volume']+1e-9)
    df['time_since_last_sell']=(df['last_transaction_time']-pd.to_datetime(df['creator_last_sell_time'],utc=True,errors='coerce')).dt.total_seconds().fillna(-1)
    df['social_score']=(
        df['ipfs_url_twitter']+df['ipfs_url_website']+df['ipfs_url_telegram']+df['ipfs_url_discord']+
        df['ipfs_tokenuri_twitter']+df['ipfs_tokenuri_website']+df['ipfs_tokenuri_telegram']+df['ipfs_tokenuri_discord']
    )

le=LabelEncoder()
all_cr=pd.concat([train_m['creator'].fillna('unknown'),test_m['creator'].fillna('unknown')]).astype(str).unique()
le.fit(all_cr)
train_m['creator_encoded']=le.transform(train_m['creator'].fillna('unknown').astype(str))
test_m ['creator_encoded']=le.transform(test_m ['creator'].fillna('unknown').astype(str))

potential_cat=['creator_encoded','launch_hour','launch_day','decimals','bundle_size']
for df in [train_m,test_m]:
    for c in potential_cat:
        if c in df.columns:
            fv=-1
            if c=='decimals': fv=0
            if c=='bundle_size': fv=1
            if pd.api.types.is_numeric_dtype(df[c]):
                df[c]=df[c].fillna(fv)
            else:
                df[c]=df[c].fillna('missing')
            try:
                df[c]=pd.to_numeric(df[c],errors='coerce').fillna(fv).astype(int)
            except:
                pass

base_features=[
    'total_sol_volume','avg_sol_volume','std_sol_volume','p90_sol_volume','max_sol_volume','min_sol_volume',
    'count_buys','count_sells','buy_ratio','unique_wallets','wallet_diversity',
    'price_change','price_volatility','max_price','min_price','avg_price',
    'tx_per_second','avg_time_diff','std_time_diff','min_time_diff','max_time_diff',
    'initial_liquidity_sol','final_liquidity_sol','liquidity_change_sol','std_liquidity_sol',
    'liquidity_volatility_ratio','price_stability','price_range','price_range_norm',
    'avg_gas_fee','total_gas_fee','avg_gas_ratio','std_gas_ratio','gas_efficiency','avg_consumed_gas','total_consumed_gas',
    'creator_token_count','creator_gas_avg',
    'launch_hour','launch_day','launch_time_gap',
    'bundle_size','gas_used','decimals',
    'creator_encoded','time_duration'
]
text_feats=[
    'name_len','symbol_len','name_word_count','name_upper_ratio','symbol_is_allcaps',
    'name_special_chars','symbol_special_chars','name_emoji_count','symbol_emoji_count','name_has_emoji','symbol_has_emoji'
]+[f'name_has_{kw}' for kw in memes]+[f'symbol_has_{kw}' for kw in memes]
meme_feats=[
    'meme_score_name','meme_score_symbol','meme_score_desc',
    'has_meme_keyword_name','has_meme_keyword_symbol','has_meme_keyword_desc','has_meme_keyword_any'
]
ipfs_feats=[
    'ipfs_url_twitter','ipfs_url_website','ipfs_url_telegram','ipfs_url_discord','ipfs_url_desc_len','ipfs_url_num_keys',
    'ipfs_tokenuri_twitter','ipfs_tokenuri_website','ipfs_tokenuri_telegram','ipfs_tokenuri_discord','ipfs_tokenuri_desc_len','ipfs_tokenuri_num_keys',
    'social_score'
]
desc_feats=['combined_desc_len','desc_word_count','desc_has_link','desc_pos_score','desc_neg_score']+[f'desc_has_{kw}' for kw in keywords_desc]
creator_activity=[
    'creator_tx_count','creator_sell_count','creator_buy_count','creator_sell_ratio',
    'creator_did_sell','creator_total_sold_sol','creator_avg_sell_sol',
    'creator_first_sell_time_diff','creator_last_sell_time_diff',
    'creator_first_sell_relative','creator_last_sell_relative',
    'creator_sell_impact','time_since_last_sell'
]
interaction=['buy_sell_ratio','unique_wallet_per_tx','sol_volume_per_wallet']
final_features=sorted(set(
    base_features+text_feats+meme_feats+ipfs_feats+desc_feats+creator_activity+interaction+tfidf_svd_cols
))

missing_train=[f for f in final_features if f not in train_m.columns]
missing_test =[f for f in final_features if f not in test_m.columns]
if missing_train: print(f"Missing in train: {missing_train}")
if missing_test:  print(f"Missing in test:  {missing_test}")
final_features=[f for f in final_features if f in train_m.columns and f in test_m.columns]

X=train_m[final_features].copy()
X_test=test_m[final_features].copy()
X.replace([np.inf,-np.inf],np.nan,inplace=True)
X_test.replace([np.inf,-np.inf],np.nan,inplace=True)
for c in final_features:
    if X[c].isnull().any():
        if pd.api.types.is_numeric_dtype(X[c]):
            m=X[c].median()
            X[c].fillna(m,inplace=True)
            X_test[c].fillna(m if c in X_test.columns else 0,inplace=True)
        else:
            m=X[c].mode()[0] if not X[c].mode().empty else 'missing'
            X[c].fillna(m,inplace=True)
            X_test[c].fillna(m,inplace=True)

const_cols=[c for c in X.columns if X[c].nunique()<=1]
if const_cols:
    X.drop(columns=const_cols,inplace=True)
    X_test.drop(columns=const_cols,inplace=True)
    final_features=[f for f in final_features if f not in const_cols]
if X.shape[1]==0:
    X['dummy_feature']=0
    X_test['dummy_feature']=0
    final_features=['dummy_feature']

y=train_m['has_graduated'].astype(int).values
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
test_preds=np.zeros(len(X_test))
oof_preds =np.zeros(len(X))

cat_pot=['creator_encoded','launch_hour','launch_day','decimals','bundle_size','creator_did_sell']
cat_feats=[c for c in cat_pot if c in X.columns]
for cf in ['creator_did_sell']:
    if cf in X.columns:
        X[cf]=X[cf].astype(str)
        X_test[cf]=X_test[cf].astype(str)

try:
    import torch
    task_type='GPU' if torch.cuda.is_available() else 'CPU'
except ImportError:
    task_type='CPU'

for fold,(train_idx,val_idx) in enumerate(skf.split(X,y)):
    X_tr,X_val=X.iloc[train_idx],X.iloc[val_idx]
    y_tr,y_val=y[train_idx],y[val_idx]
    model=catboost.CatBoostClassifier(
        iterations=10000,
        learning_rate=0.03,
        depth=8,
        eval_metric='Logloss',
        loss_function='Logloss',
        early_stopping_rounds=150,
        random_seed=42+fold,
        verbose=200,
        task_type=task_type
    )
    model.fit(
        X_tr,y_tr,
        eval_set=(X_val,y_val),
        use_best_model=True,
        cat_features=cat_feats
    )
    oof_preds[val_idx]=model.predict_proba(X_val)[:,1]
    test_preds+=model.predict_proba(X_test[X_tr.columns])[:,1]/skf.n_splits

oof_logloss=-np.mean(y*np.log(oof_preds+1e-15)+(1-y)*np.log(1-oof_preds+1e-15))
oof_auc=roc_auc_score(y,oof_preds)
print(f"OOF ROC-AUC: {oof_auc:.5f}, LogLoss: {oof_logloss:.5f}")

submission=pd.DataFrame({'mint':test_m['mint'],'has_graduated':test_preds})
if submission['mint'].duplicated().any():
    submission=submission.groupby('mint',as_index=False).mean()
submission['has_graduated'].fillna(0.5,inplace=True)
submission.to_csv('submission.csv',index=False)
print("Submission.csv created")


Loading data...
Data loaded.
Processing metadata...
Metadata processed.
Processing transaction chunks...


Processing chunks:   0%|          | 0/41 [00:00<?, ?it/s]

Transaction features processed.
Merging features...
Features merged.
Starting IPFS metadata retrieval...
Token URLs...
Processed 43453/869061 URLs (5.0%) in 44.39s
Processed 86906/869061 URLs (10.0%) in 75.17s
Processed 130359/869061 URLs (15.0%) in 106.45s
Processed 173812/869061 URLs (20.0%) in 137.84s
Processed 217265/869061 URLs (25.0%) in 169.03s
Processed 260718/869061 URLs (30.0%) in 201.75s
Processed 304171/869061 URLs (35.0%) in 230.54s
Processed 347624/869061 URLs (40.0%) in 258.26s
Processed 391077/869061 URLs (45.0%) in 288.10s
Processed 434530/869061 URLs (50.0%) in 316.17s
Processed 477983/869061 URLs (55.0%) in 343.13s
Processed 521436/869061 URLs (60.0%) in 370.23s
Processed 564889/869061 URLs (65.0%) in 396.60s
Processed 608342/869061 URLs (70.0%) in 425.19s
Processed 651795/869061 URLs (75.0%) in 453.34s
Processed 695248/869061 URLs (80.0%) in 484.31s
Processed 738701/869061 URLs (85.0%) in 511.25s
Processed 782154/869061 URLs (90.0%) in 539.26s
Processed 825607/86906

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:550: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(
<ipython-input-1-222da9cbb174>:390: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_meme_keyword_symbol']=df['meme_score_symbol'].gt(0).astype(int)
<ipython-input-1-222da9cbb174>:391: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_meme_keyword_desc']=df['meme_score_desc'].gt(0).astype(int)
<ipython-input-1-222da9cbb174>:392: PerformanceWa

0:	learn: 0.5835032	test: 0.5834170	best: 0.5834170 (0)	total: 248ms	remaining: 41m 15s
200:	learn: 0.0186886	test: 0.0198696	best: 0.0198691 (199)	total: 10.4s	remaining: 8m 26s
400:	learn: 0.0170690	test: 0.0195060	best: 0.0195060 (400)	total: 20.5s	remaining: 8m 10s
600:	learn: 0.0158224	test: 0.0193777	best: 0.0193756 (596)	total: 30.7s	remaining: 7m 59s
800:	learn: 0.0147574	test: 0.0192789	best: 0.0192783 (793)	total: 40.6s	remaining: 7m 45s
1000:	learn: 0.0138294	test: 0.0192068	best: 0.0192068 (1000)	total: 50.5s	remaining: 7m 34s
1200:	learn: 0.0129742	test: 0.0191737	best: 0.0191728 (1102)	total: 1m	remaining: 7m 26s
1400:	learn: 0.0121647	test: 0.0191852	best: 0.0191488 (1257)	total: 1m 11s	remaining: 7m 17s
bestTest = 0.0191488059
bestIteration = 1257
Shrink model to first 1258 iterations.
0:	learn: 0.5784084	test: 0.5783335	best: 0.5783335 (0)	total: 110ms	remaining: 18m 22s
200:	learn: 0.0184860	test: 0.0207373	best: 0.0207373 (200)	total: 10.2s	remaining: 8m 19s
400:	lea

<ipython-input-1-222da9cbb174>:553: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  submission['has_graduated'].fillna(0.5,inplace=True)


Submission.csv created
